In [ ]:
%pip install ta
import pandas as pd
import numpy as np
import os
import ta
from sklearn.preprocessing import StandardScaler

# --- הגדרת המטבע ---
SYMBOL = 'BTCUSDT'

# --- 1. חיבור לגוגל דרייב ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
    print("✅ מחובר לגוגל קולאב!")
except ImportError:
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()
    print(f"💻 מריץ בסביבה מקומית. תיקיית הבסיס: {BASE_DIR}")

SEQ_LENGTH = 120
PREDICT_AHEAD = 1
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
STEP = 5

FEATURE_COLS = [
    'log_ret', 'rsi', 'rsi_change', 'rsi_accel', 'macd', 'macd_slope',
    'bb_pband_change', 'volume', 'ma_dist', 'volume_z', 'vol_spike', 'adx',
    'hour_sin', 'hour_cos'
]

# המטרה עבור LSTM
TARGET_COL = 'target_15m'

class DataPreprocessorLSTM:
    def load_and_clean_data(self, filepath):
        print(f"Loading data from {filepath}...")
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"❌ לא נמצא קובץ נתונים בנתיב:\n{filepath}\nוודא שקוד ה-API סיים לרוץ.")

        df = pd.read_csv(filepath)
        df['open_time'] = pd.to_datetime(df['open_time'])
        df = df.sort_values('open_time').drop_duplicates(subset=['open_time']).ffill().dropna()

        # חישובי פיצ'רים
        df['log_ret'] = np.log((df['close'] / df['close'].shift(1))* 10)
        df['rsi'] = ta.momentum.rsi(df['close'], window=14) / 100.0
        df['rsi_change'] = df['rsi'].diff(periods=3)
        df['rsi_accel'] = df['rsi_change'].diff(periods=2)

        macd = ta.trend.MACD(df['close'])
        macd_raw = macd.macd_diff()
        df['macd'] = (macd_raw - macd_raw.rolling(window=100).mean()) / (macd_raw.rolling(window=100).std() + 1e-9)
        df['macd_diff'] = macd.macd_diff()
        df['macd_slope'] = df['macd_diff'].diff(periods=2)

        bb = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
        df['bb_pband'] = bb.bollinger_pband()
        df['bb_pband_change'] = df['bb_pband'].diff(periods=1)

        df['volume'] = np.log(df['volume'] + 1)
        df['volume'] = (df['volume'] - df['volume'].mean()) / (df['volume'].std() + 1e-9)
        df['vol_ma'] = df['volume'].rolling(window=20).mean()
        df['vol_std'] = df['volume'].rolling(window=20).std()
        df['volume_z'] = (df['volume'] - df['vol_ma']) / (df['vol_std'] + 1e-9)
        df['vol_spike'] = (df['volume'] > (df['vol_ma'] * 2)).astype(float)

        df['ma_20'] = df['close'].rolling(window=20).mean()
        df['ma_dist'] = (df['close'] - df['ma_20']) / (df['ma_20'] + 1e-9) * 10.0

        df['target_15m'] = np.log(df['close'].shift(-3) / df['close'])

        adx = ta.trend.ADXIndicator(df['high'], df['low'], df['close'], window=14)
        df['adx'] = adx.adx()

        df['hour'] = df['open_time'].dt.hour
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

        return df.dropna()

    def create_sequences(self, data, target, seq_length, step=1):
        xs, ys = [], []
        for i in range(0, len(data) - seq_length - PREDICT_AHEAD + 1, step):
            xs.append(data[i : (i + seq_length)])
            ys.append(target[i + seq_length + PREDICT_AHEAD - 1])
        return np.array(xs), np.array(ys)

    def process(self):
        # Pathing updated to match the ETH code dynamic structure
        data_path = os.path.join(BASE_DIR, 'data', f'{SYMBOL}_5m_data.csv')

        try:
            df = self.load_and_clean_data(data_path)
        except FileNotFoundError as e:
            print(e)
            return

        # Using float32 to prevent memory crashes and halve file size!
        data = df[FEATURE_COLS].values.astype(np.float32)
        target = df[TARGET_COL].values.astype(np.float32)

        n = len(data)
        train_end = int(n * TRAIN_SPLIT)
        val_end = int(n * (TRAIN_SPLIT + VAL_SPLIT))

        scaler = StandardScaler()
        train_data_scaled = scaler.fit_transform(data[:train_end]).astype(np.float32)
        val_data_scaled = scaler.transform(data[train_end:val_end]).astype(np.float32)
        test_data_scaled = scaler.transform(data[val_end:]).astype(np.float32)

        print(f"\nCreating sliding windows (SEQ_LENGTH={SEQ_LENGTH}, Features={len(FEATURE_COLS)}, Step={STEP})...")
        X_train, y_train = self.create_sequences(train_data_scaled, target[:train_end], SEQ_LENGTH, STEP)
        X_val, y_val = self.create_sequences(val_data_scaled, target[train_end:val_end], SEQ_LENGTH, STEP)
        X_test, y_test = self.create_sequences(test_data_scaled, target[val_end:], SEQ_LENGTH, STEP)

        # Pathing updated to match the ETH code dynamic structure
        save_dir = os.path.join(BASE_DIR, 'processed_data_lstm', SYMBOL)
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n💾 מתחיל שמירה לתיקייה:\n{save_dir}")

        np.save(os.path.join(save_dir, 'X_train.npy'), X_train)
        np.save(os.path.join(save_dir, 'y_train.npy'), y_train)
        np.save(os.path.join(save_dir, 'X_val.npy'), X_val)
        np.save(os.path.join(save_dir, 'y_val.npy'), y_val)
        np.save(os.path.join(save_dir, 'X_test.npy'), X_test)
        np.save(os.path.join(save_dir, 'y_test.npy'), y_test)
        print(f"✅ Preprocessing Complete for {SYMBOL} (LSTM). Saved to {save_dir}")

if __name__ == "__main__":
    DataPreprocessorLSTM().process()

    # Optional but highly recommended: Flush data to Drive explicitly if running in Colab


Note: you may need to restart the kernel to use updated packages.
💻 מריץ בסביבה מקומית. תיקיית הבסיס: c:\Users\ofich\cryptoProj\CryptoProject\CryptoProject
Loading data from c:\Users\ofich\cryptoProj\CryptoProject\CryptoProject\data\BTCUSDT_5m_data.csv...

Creating sliding windows (SEQ_LENGTH=120, Features=14, Step=5)...

💾 מתחיל שמירה לתיקייה:
c:\Users\ofich\cryptoProj\CryptoProject\CryptoProject\processed_data_lstm\BTCUSDT
✅ Preprocessing Complete for BTCUSDT (LSTM). Saved to c:\Users\ofich\cryptoProj\CryptoProject\CryptoProject\processed_data_lstm\BTCUSDT


In [ ]:
import pandas as pd
import numpy as np
import os
import ta
import shutil
from sklearn.preprocessing import StandardScaler

# --- הגדרת המטבע ---
SYMBOL = 'BTCUSDT'

# --- 1. חיבור לגוגל דרייב ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
    print("✅ מחובר לגוגל קולאב!")
except ImportError:
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()
    print(f"💻 מריץ בסביבה מקומית. תיקיית הבסיס: {BASE_DIR}")

SEQ_LENGTH = 120
PREDICT_AHEAD = 1
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
STEP = 5

FEATURE_COLS = [
    'log_ret', 'rsi', 'rsi_change', 'rsi_accel', 'macd', 'macd_slope',
    'bb_pband_change', 'volume', 'ma_dist', 'volume_z', 'vol_spike', 'adx',
    'hour_sin', 'hour_cos'
]

TARGET_COL = 'target_15m'

class DataPreprocessorLSTM:
    def load_and_clean_data(self, filepath):
        print(f"Loading data from {filepath}...")
        df = pd.read_csv(filepath)
        df['open_time'] = pd.to_datetime(df['open_time'])
        df = df.sort_values('open_time').drop_duplicates(subset=['open_time']).ffill().dropna()

        df['log_ret'] = np.log((df['close'] / df['close'].shift(1))* 10)
        df['rsi'] = ta.momentum.rsi(df['close'], window=14) / 100.0
        df['rsi_change'] = df['rsi'].diff(periods=3)
        df['rsi_accel'] = df['rsi_change'].diff(periods=2)

        macd = ta.trend.MACD(df['close'])
        macd_raw = macd.macd_diff()
        df['macd'] = (macd_raw - macd_raw.rolling(window=100).mean()) / (macd_raw.rolling(window=100).std() + 1e-9)
        df['macd_diff'] = macd.macd_diff()
        df['macd_slope'] = df['macd_diff'].diff(periods=2)

        bb = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
        df['bb_pband'] = bb.bollinger_pband()
        df['bb_pband_change'] = df['bb_pband'].diff(periods=1)

        df['volume'] = np.log(df['volume'] + 1)
        df['volume'] = (df['volume'] - df['volume'].mean()) / (df['volume'].std() + 1e-9)
        df['vol_ma'] = df['volume'].rolling(window=20).mean()
        df['vol_std'] = df['volume'].rolling(window=20).std()
        df['volume_z'] = (df['volume'] - df['vol_ma']) / (df['vol_std'] + 1e-9)
        df['vol_spike'] = (df['volume'] > (df['vol_ma'] * 2)).astype(float)

        df['ma_20'] = df['close'].rolling(window=20).mean()
        df['ma_dist'] = (df['close'] - df['ma_20']) / (df['ma_20'] + 1e-9) * 10.0

        df['target_15m'] = np.log(df['close'].shift(-3) / df['close'])

        adx = ta.trend.ADXIndicator(df['high'], df['low'], df['close'], window=14)
        df['adx'] = adx.adx()

        df['hour'] = df['open_time'].dt.hour
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

        return df.dropna()

    def create_sequences(self, data, target, seq_length, step=1):
        xs, ys = [], []
        for i in range(0, len(data) - seq_length - PREDICT_AHEAD + 1, step):
            xs.append(data[i : (i + seq_length)])
            ys.append(target[i + seq_length + PREDICT_AHEAD - 1])
        return np.array(xs), np.array(ys)

    def process(self):
        data_path = os.path.join(BASE_DIR, 'data', f'{SYMBOL}_5m_data.csv')

        try:
            df = self.load_and_clean_data(data_path)
        except FileNotFoundError as e:
            print(e)
            return

        data = df[FEATURE_COLS].values.astype(np.float32)
        target = df[TARGET_COL].values.astype(np.float32)

        n = len(data)
        train_end = int(n * TRAIN_SPLIT)
        val_end = int(n * (TRAIN_SPLIT + VAL_SPLIT))

        scaler = StandardScaler()
        # We still need to fit the scaler to keep the math accurate
        train_data_scaled = scaler.fit_transform(data[:train_end]).astype(np.float32)
        val_data_scaled = scaler.transform(data[train_end:val_end]).astype(np.float32)
        test_data_scaled = scaler.transform(data[val_end:]).astype(np.float32)

        print(f"\nCreating sliding windows... (Calculating X_test)")
        # We only strictly need to create X_test sequences right now!
        X_test, y_test = self.create_sequences(test_data_scaled, target[val_end:], SEQ_LENGTH, STEP)

        save_dir = os.path.join(BASE_DIR, 'processed_data_lstm', SYMBOL)
        os.makedirs(save_dir, exist_ok=True)

        print("\n💾 מתחיל שמירה בטוחה של הקובץ החסר...")

        # 1. Save to Colab's local high-speed memory FIRST
        local_temp_path = '/content/X_test.npy'
        print("1/2: Saving X_test.npy locally to A100 SSD...")
        np.save(local_temp_path, X_test)

        # 2. Force a secure file copy over to Google Drive
        drive_path = os.path.join(save_dir, 'X_test.npy')
        print("2/2: Transferring X_test.npy to Google Drive (Please wait)...")
        shutil.copy(local_temp_path, drive_path)

        print(f"\n✅ הקובץ החסר הושלם וסונכרן בהצלחה לדרייב: {drive_path}")

if __name__ == "__main__":
    DataPreprocessorLSTM().process()

💻 מריץ בסביבה מקומית. תיקיית הבסיס: c:\Users\ofich\cryptoProj\CryptoProject\CryptoProject
Loading data from c:\Users\ofich\cryptoProj\CryptoProject\CryptoProject\data\BTCUSDT_5m_data.csv...

Creating sliding windows... (Calculating X_test)

💾 מתחיל שמירה בטוחה של הקובץ החסר...
1/2: Saving X_test.npy locally to A100 SSD...
2/2: Transferring X_test.npy to Google Drive (Please wait)...

✅ הקובץ החסר הושלם וסונכרן בהצלחה לדרייב: c:\Users\ofich\cryptoProj\CryptoProject\CryptoProject\processed_data_lstm\BTCUSDT\X_test.npy
